---
title: Single-cell Elbowplot
author: Zafer Kosar
format:
    html:
        code-fold: true
        code-summary: "Show code"
---

In [ ]:
# Core scverse libraries
import anndata as ad

# data manipulation
import polars as pl
import scanpy as sc

# ggplot but interactive and python
from lets_plot import (
    LetsPlot,
    aes,
    element_blank,
    element_line,
    element_text,
    geom_blank,
    geom_hline,
    geom_jitter,
    geom_label,
    geom_line,
    geom_point,
    geom_smooth,
    geom_text,
    geom_violin,
    gggrid,
    ggplot,
    ggsize,
    ggtb,
    guide_colorbar,
    guides,
    labs,
    layer_tooltips,
    scale_color_continuous,
    scale_color_gradient,
    scale_color_hue,
    scale_color_viridis,
    scale_x_continuous,
    theme,
    theme_classic,
)

LetsPlot.setup_html()

from typing import TYPE_CHECKING, Literal

import numpy as np
from lets_plot.plot.core import PlotSpec
from scanpy import AnnData
from scipy.optimize import curve_fit

In [ ]:
# read the sampel data
adata = sc.read("data/pbmc3k_pped.h5ad")

In [ ]:
adata

In [ ]:
def _exp_decay(x, a, b):
    return a * np.exp(-x * b)


def _pc_fit(df: pl.DataFrame) -> pl.DataFrame:
    values = df.select("variance").to_numpy().flatten()
    x = df.select("PC").to_numpy().flatten()
    popt, _ = curve_fit(_exp_decay, x, values)
    fit = _exp_decay(x, *popt)
    mean_lifetime_point_sqr = np.max(fit) / (2.71828**2)
    # find for what x , y is mean_lifetime_point_sqr
    a, b = popt
    x_intercept = np.log(mean_lifetime_point_sqr / a) / -b
    return df.with_columns(pl.Series(_exp_decay(x, *popt)).alias("exp_fit")), x_intercept

In [ ]:
_THEME_ELBOW = (
    theme_classic()
    + theme(
        text=element_text(color="#1f1f1f", family="Arial", size=12),
        axis_text_x=element_text(color="#1f1f1f", family="Arial", size=14),
        axis_text_y=element_text(color="#1f1f1f", family="Arial", size=14),
        axis_title=element_text(color="#1f1f1f", family="Arial", size=18),
    )
    + labs(y="Variance")
    + ggsize(600, 400)
)


In [ ]:
def elbow(
    data: AnnData,
    n_pcs: int = 50,
    *,
    scale: Literal["log", "linear"] = "log",
    fit: bool = True,
    line_size: int = 2,
    label: bool = True,
    interactive: bool = False,
    color_hline: str = "#3f3f3f",
    color_point: str = "#6f6f6f",
    color_line: str = "#d26868",
    shadow: bool = True,
    hline: bool = True,
    **point_kwargs,
) -> PlotSpec:

    # handle the data type
    if not isinstance(data, AnnData):
        msg = "data must be an AnnData object"
        raise TypeError(msg)

    # Sub sample the data
    col_names = [f"{i+1}" for i in range(n_pcs)]
    # get the PCs from the anndata object
    frame = pl.from_numpy(data.obsm["X_pca"][:, :n_pcs], schema=col_names)
    # Calculate the variance explained by each PC, transpose, and rename the columns
    frame = (
        frame.select(pl.all().var())
        .transpose(include_header=True, header_name="PC", column_names=["variance"])
        .with_columns(pl.col("PC").cast(pl.Int16))
    )

    # Handle the scale
    if scale == "log":
        frame = frame.with_columns(pl.col("variance").log())
    elif scale == "linear":
        pass
    else:
        raise ValueError("scale must be either 'log' or 'linear'")

    # Create the plot
    elbw = (
        ggplot(data=frame)
        + geom_point(
            aes(x="PC", y="variance"),
            size=5,
            color=color_point,
            **point_kwargs,
        )
        + _THEME_ELBOW
    )

    # handle the fit
    if fit:
        frame, x_intercept = _pc_fit(frame)
        # handle the shadow
        if shadow:
            elbw += geom_line(
                data=frame,
                mapping=aes(x="PC", y="exp_fit"),
                size=line_size * 2,
                color=color_line,
                alpha=0.2,
            )
        elbw += geom_line(
            data=frame, mapping=aes(x="PC", y="exp_fit"), size=line_size, color=color_line
        )
        # handle the hline
        if hline:
            mean_lifetime_point_sqr = frame.select("exp_fit").max().item() / (2.71828**2)
            elbw += geom_hline(
                yintercept=mean_lifetime_point_sqr, color=color_hline, size=1, linetype="dashed"
            )
        # handle the label
        if label:
            elbw += geom_label(
                hjust=0.5,
                yjust=0.5,
                label=f"X intercept = {x_intercept:.2f}",
                color="#3f3f3f",
                size=8,
                x=x_intercept,
                fontface="bold",
            )

    if interactive:
        elbw += ggtb()

    return elbw

In [ ]:
elbow(adata, scale="log", n_pcs=40, color_hline="#3f3f3f")

In [ ]:
n_pcs = 40
col_names = [f"{i+1}" for i in range(n_pcs)]
frame = pl.from_numpy(adata.obsm["X_pca"][:, :n_pcs], schema=col_names)

In [ ]:
frame = (
    frame.select(pl.all().var())
    .transpose(include_header=True, header_name="PC", column_names=["variance"])
    .with_columns(pl.col("PC").cast(pl.Int16))
)
frame